In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)


# 1. Load cleaned sources

metadata = pd.read_csv(PROCESSED_DIR / "metadata_clean.csv")
text = pd.read_csv(PROCESSED_DIR / "text_clean.csv")
audio = pd.read_csv(PROCESSED_DIR / "audio_clean.csv")
video = pd.read_csv(PROCESSED_DIR / "video_clean.csv")

print("metadata:", metadata.shape)
print("text:    ", text.shape)
print("audio:   ", audio.shape)
print("video:   ", video.shape)

metadata: (5136, 21)
text:     (5136, 513)
audio:    (5136, 4097)
video:    (5136, 2305)


In [ ]:

# 2. Validate before merging


# 2a. id_hash must be unique within each file
for name, df in [("metadata", metadata), ("text", text), ("audio", audio), ("video", video)]:
    dupes = df["id_hash"].duplicated().sum()
    print(f"{name:10s} unique id_hash: {df['id_hash'].nunique():5d} / {len(df):5d} rows   duplicates: {dupes}")
    assert dupes == 0, f"{name}_clean.csv has duplicate id_hash values, resolve before merging"

# 2b. The four ID sets should match
ids_meta = set(metadata["id_hash"])
ids_text = set(text["id_hash"])
ids_audio = set(audio["id_hash"])
ids_video = set(video["id_hash"])

common_ids = ids_meta & ids_text & ids_audio & ids_video
print("IDs common to all 4 sources:", len(common_ids))

for name, ids in [("metadata", ids_meta), ("text", ids_text), ("audio", ids_audio), ("video", ids_video)]:
    missing = ids - common_ids
    if missing:
        print(f"  {name}: {len(missing)} id_hash values NOT present in all other sources (will be dropped)")
    else:
        print(f"  {name}: fully consistent with the other three sources")

# 2c. No leftover NaNs/Infs in the embedding files
for name, df in [("text", text), ("audio", audio), ("video", video)]:
    n_nan = df.drop(columns=["id_hash"]).isna().sum().sum()
    n_inf = np.isinf(df.drop(columns=["id_hash"]).to_numpy(dtype=float)).sum()
    print(f"{name:6s} embeddings -- NaNs: {n_nan}   Infs: {n_inf}")
    assert n_nan == 0 and n_inf == 0, f"{name}_clean.csv still has invalid values, fix in cleaning step first"

metadata   unique id_hash:  5136 /  5136 rows   duplicates: 0
text       unique id_hash:  5136 /  5136 rows   duplicates: 0
audio      unique id_hash:  5136 /  5136 rows   duplicates: 0
video      unique id_hash:  5136 /  5136 rows   duplicates: 0
IDs common to all 4 sources: 5136
  metadata: fully consistent with the other three sources
  text: fully consistent with the other three sources
  audio: fully consistent with the other three sources
  video: fully consistent with the other three sources
text   embeddings -- NaNs: 0   Infs: 0


audio  embeddings -- NaNs: 0   Infs: 0
video  embeddings -- NaNs: 0   Infs: 0


In [ ]:

# 3. Merge


# Canonical ID order = metadata rows restricted to the common set, sorted for reproducibility
id_order = metadata.loc[metadata["id_hash"].isin(common_ids), "id_hash"].sort_values().reset_index(drop=True)
print("Final merged sample size:", len(id_order))

def reorder_embeddings(df, id_order):
    """Return embedding matrix (id_hash dropped) reordered to match id_order exactly."""
    df_indexed = df.set_index("id_hash")
    aligned = df_indexed.loc[id_order]
    return aligned.to_numpy(dtype=np.float32)

text_mat = reorder_embeddings(text, id_order)
audio_mat = reorder_embeddings(audio, id_order)
video_mat = reorder_embeddings(video, id_order)

print("text_mat: ", text_mat.shape)
print("audio_mat:", audio_mat.shape)
print("video_mat:", video_mat.shape)
assert text_mat.shape[0] == audio_mat.shape[0] == video_mat.shape[0] == len(id_order)

metadata_merged = (
    metadata[metadata["id_hash"].isin(common_ids)]
    .set_index("id_hash")
    .loc[id_order]
    .reset_index()
)
assert (metadata_merged["id_hash"].values == id_order.values).all(), "Row order mismatch between metadata and embeddings!"

Final merged sample size: 5136
text_mat:  (5136, 512)
audio_mat: (5136, 4096)
video_mat: (5136, 2304)


In [ ]:

# 4. Feature engineering (log-transforms)

for col in ["plays", "likes", "comments", "shares"]:
    if f"log_{col}" not in metadata_merged.columns:
        metadata_merged[f"log_{col}"] = np.log1p(metadata_merged[col])

if "log_followers" not in metadata_merged.columns:
    metadata_merged["log_followers"] = np.log1p(metadata_merged["author_followers"])

In [ ]:

# 5. Save merged outputs

metadata_merged.to_csv(RESULTS_DIR / "analytical_dataset.csv", index=False)
np.save(RESULTS_DIR / "text_embeddings_aligned.npy", text_mat)
np.save(RESULTS_DIR / "audio_embeddings_aligned.npy", audio_mat)
np.save(RESULTS_DIR / "video_embeddings_aligned.npy", video_mat)
np.save(RESULTS_DIR / "id_order.npy", id_order.to_numpy())

print("Saved:")
print(" ", RESULTS_DIR / "analytical_dataset.csv", "-", metadata_merged.shape)
print(" ", RESULTS_DIR / "text_embeddings_aligned.npy", "-", text_mat.shape)
print(" ", RESULTS_DIR / "audio_embeddings_aligned.npy", "-", audio_mat.shape)
print(" ", RESULTS_DIR / "video_embeddings_aligned.npy", "-", video_mat.shape)

Saved:
  ..\results\analytical_dataset.csv - (5136, 26)
  ..\results\text_embeddings_aligned.npy - (5136, 512)
  ..\results\audio_embeddings_aligned.npy - (5136, 4096)
  ..\results\video_embeddings_aligned.npy - (5136, 2304)


In [ ]:

# 6. Final sanity check

check = pd.read_csv(RESULTS_DIR / "analytical_dataset.csv")
check_text = np.load(RESULTS_DIR / "text_embeddings_aligned.npy")
check_ids = np.load(RESULTS_DIR / "id_order.npy", allow_pickle=True)

assert len(check) == check_text.shape[0] == len(check_ids)
assert (check["id_hash"].values == check_ids).all()
print(f"All checks passed. Final merged N = {len(check)}")

All checks passed. Final merged N = 5136
